In [1]:
import pandas as pd
import numpy as np
from sklearn import datasets
import plotly.express as px
import plotly.graph_objs as go
from dash import Dash, dcc, html, Input, Output

# <center> Логистическая регрессия </center>

In [2]:
iris = datasets.load_iris()

In [3]:
print(iris.DESCR)

.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

    :Number of Instances: 150 (50 in each of three classes)
    :Number of Attributes: 4 numeric, predictive attributes and the class
    :Attribute Information:
        - sepal length in cm
        - sepal width in cm
        - petal length in cm
        - petal width in cm
        - class:
                - Iris-Setosa
                - Iris-Versicolour
                - Iris-Virginica
                
    :Summary Statistics:

    ============== ==== ==== ======= ===== ====================
                    Min  Max   Mean    SD   Class Correlation
    ============== ==== ==== ======= ===== ====================
    sepal length:   4.3  7.9   5.84   0.83    0.7826
    sepal width:    2.0  4.4   3.05   0.43   -0.4194
    petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
    petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
    ============== ==== ==== ======= ===== ===========

In [4]:
df = pd.DataFrame(iris.data, columns = iris.feature_names)

In [5]:
df['target'] = [1 if x==0 else -1 for x in iris.target]

In [6]:
app = Dash(__name__)


app.layout = html.Div([
    html.H4('Интерактивная диаграмма рассеяния'),
    html.P('Признак 1:'),
    dcc.Dropdown(df.columns, df.columns[0], id='dropdown_value_1'),
    dcc.Dropdown(df.columns, df.columns[1], id='dropdown_value_2'),
    dcc.Graph(id="graph"),
      
])


@app.callback(
    Output("graph", "figure"), 
    [Input("dropdown_value_1", "value"),
     Input("dropdown_value_2", "value")])
def display_hist(dropdown_value_1, dropdown_value_2):
    fig = px.scatter(df, x = dropdown_value_1, y = dropdown_value_2, color = 'target')
    return fig

app.run(debug=True)

## Задача бинарной классификации

`Дано:` 
- множество объектов обучающей выборки $X^l = (x_i, y_i)_{i=1}^{l}, x_i\in \mathbb{R}$, разделенных на классы $Y=\{-1,1\}$; 
- множество объектов тестовой выборки, классовая принадлежность которых не известна. 


### 1. Линейная модель

#### Регрессия

Обучающая выборка $X^l = (x_i, y_i)_{i=1}^{l}, \quad x_i\in \mathbb{R}^n, \quad y_i = y (x_i)\in \mathbb{R}$

1. Модель регрессии -- *линейная* с параметром $w\in \mathbb{R}^n$:

$$a(x,w ) = \langle x, w \rangle = \sum_{j=1}^n w_j f_j(x)$$

2. Функция потерь -- *квадратичная*:

$$\mathcal{L}(w, x) = (a(x,w) - y(x))^2$$

3. Метод обучения -- *метод наименьших квадратов*:

$$Q(w) = \sum_{i=1}^l\mathcal{L}(w,x_i) = \sum_{i=1}^l \left( a(x,w) - y(x) \right) ^2 \rightarrow \underset{w}{min}$$

4. Проверка на тестовой выборке $X^k = (\tilde{x_i}, \tilde{y_i})_{i=1}^k$:

$$\tilde{Q}(w) = \frac{1}{k}\sum_{i=1}^l(a(x,w) - y(x))^2 $$

#### Классификация

Обучающая выборка $X^l = (x_i, y_i)_{i=1}^{l}, \quad x_i\in \mathbb{R}^n, \quad y_i \in \{-1, 1\}$

1. Модель классификации -- *линейная* с параметром $w\in \mathbb{R}^n$:

$$a(x,w ) = sign \langle x, w \rangle = sign \sum_{j=1}^n w_j f_j(x)$$

2. Функция потерь -- *бинарная* или ее аппроксимация:

$$\mathcal{L}(w, x) = [a(x,w)y  (x)<0] = [\langle x, w \rangle y(x) < 0] \leq L(\langle x, w \rangle y(x))$$

3. Метод обучения -- *минимизация эмпирического риска*:

$$Q(w) = \sum_{i=1}^l\mathcal{L}(w,x_i) = \sum_{i=1}^l[\langle x_i, w \rangle y_i < 0] \leq \sum_{i=1}^lL(\langle x_i, w \rangle y_i) \rightarrow \underset{w}{min}$$

4. Проверка на тестовой выборке $X^k = (\tilde{x_i}, \tilde{y_i})_{i=1}^k$:

$$\tilde{Q}(w) = \frac{1}{k}\sum_{i=1}^l[\langle x_i, w \rangle y_i < 0] $$

### Бинарный классификатор

Бинарный классификатор: $a(x,w ) = sign\,\mathit{g}(x, w), \quad Y\in\{-1, 1\}$

- $g(x,w)$ -- *разделяющая (дискриминантная) функция*
- $x: g(x,w) =0 $ -- разделяющая гиперплоскость между классами
- $M_i(w) = g(x_i, w)y_i$ -- отступ объекта $x_i$ от разделяющей поверхности, если $M_i(w)<0 \Leftrightarrow$ алгоритм $a(x,w)$ ошибается на $x_i$;
- логарифмическая функция потерь $L(\langle x_i, w \rangle y_i)  = log_2(1+e^{-\langle x_i, w \rangle y_i})$

In [7]:
import math

In [8]:
x = np.array(np.arange(-3, 3, 0.01))
y_1 = [1 if x <
       0 else 0 for x in x]
y_2 = [math.log(i, 2) for i in 1. + np.exp(-x)]
#y_2 = 2. / (1. + np.exp(x))
fig = go.Figure(data = [go.Line(x= x, y = y_1, name = 'Пороговая функция потерь'), 
                        go.Line(x= x, y = y_2, name = 'Логарифмическая функция потерь')], 
                layout = {'title' : 'Аппроксимация пороговой функции потерь'})
fig.show()

d:\Anaconda\Lib\site-packages\plotly\graph_objs\_deprecations.py:378: DeprecationWarning:

plotly.graph_objs.Line is deprecated.
Please replace it with one of the following more specific types
  - plotly.graph_objs.scatter.Line
  - plotly.graph_objs.layout.shape.Line
  - etc.




### Градиентный спуск

**Вход:** выборка $X^l$, темп обучения $h$, темп забывания $\lambda$.  
**Выход:** вектор весов $w$

**Шаги:**
1. иницализировать веса $w_j, j=0,...,n$.
2. инициализировать $Q(w) = \sum_{i=1}^lL(\langle x_i, w \rangle y_i)$
- *повторять пока значение $Q$ не стабилизируется и/или веса $w$ не перестанут изменяться:*  
3. выбрать объект $x_i$ из $X^l$;
4. сделать шаг градиентного спуска: $w = w - h L'(\langle x_i, w \rangle y_i)$
5. вычислить потерю: $\epsilon_i = L(\langle x_i, w \rangle y_i)$
6. обновить значение функционала: $Q = (1-\lambda)Q +\lambda\epsilon_i$

**Функционал в логистической регрессии:** $Q(w) = \sum_{i=1}^l log_2(1+e^{-\langle x_i, w \rangle })\rightarrow \underset{w}{min}$

**Шаг градиентного спуска:** $w = w + h x_i y_i \sigma(-\langle x_i, w \rangle y_i)$, где $\sigma(z) = \frac{1}{1+e^{-z}}$

### Результат решения задачи

In [9]:
y_1 = np.array(np.arange(0.01, 0.99, 0.1))
y_2 = y_1/(1-y_1)
y_3 = np.log(y_2)
#y_2 = 2. / (1. + np.exp(x))
fig = go.Figure(data = [go.Line(x= y_1, y = y_1, name = 'Вероятность'), 
                        go.Line(x= y_1, y = y_2, name = 'Отношение шансов'),
                       go.Line(x= y_1, y = y_3, name = 'Логарифм шансов')])
fig.show()

Логарифмические шансы: $\log(\frac{p}{1-p})$

Обозначим $b_0 + b_1 \times x_{i,1} + b_2 \times x_{i,2} +... +b_p \times x_{i,p} + \epsilon_i$, как скалярное произведение $	
\langle b, x_i \rangle$, тогда:

$$\langle b, x_i \rangle = \log(\frac{p}{1-p})$$

$$e^{\langle b, x_i \rangle} = \frac{p}{1-p}$$

$$p=\frac{1}{1+e^{-\langle b, x_i \rangle}}$$

$\sigma(z) = \frac{1}{1+e^{-z}}$ - логистическая функция (сигмоида).

### Метрики качества в задаче классификации

#### Матрица ошибок

<!-- -->   | $a(x) = -1$ | $ a(x)=1 $
--- | --- | ---
$y=-1$ | Истинно-отрицательный **(True Negative - TN)** | Ложно-положительный **(False Positive - FP)**
$y= 1$ | Ложно-отрицательный **(False Negative - FN)**| Истинно-положительный **(True Positive - TP)** 

#### Правильность классификации

$$accuracy = \frac{1}{l}\sum_{i=1}^l[a(x_i)=y_i]=\frac{TP+TN}{TP+FP+FN+TN} $$

#### Точность -  доля релевантных ответов, среди найденных.

$$precision= \frac{TP}{TP+FP}$$

#### Полнота - доля найденных среди релевантных (чувствительность)

$$recall (TPR)= \frac{TP}{TP+FN}$$

Доля верных положительных ответов.

#### F1-мера - гармоническое среднее между точностью и полнотой

$$F1 = \frac{2*precision*recall}{precision+recall}$$

#### Логистическая функция потерь

$$logloss = -\frac1l\sum_{i=1}^l(y_i*log(p_i) + (1-y_i)log(1-p_i)), $$

где $p_i$ -- вероятность принадлежности объекта к классу 1.

#### ROC-AUC -- площадь под кривой ошибок

Для каждого значения порога классификации: от 0 до 1, определяются величины TPR и FPR, которые и отображаются на графике.

$$FPR = \frac{FP}{FP+TN}$$

Площадь под кривой в данном случае показывает качество алгоритма (больше — лучше), кроме этого, важной является крутизна самой кривой — мы хотим максимизировать TPR, минимизируя FPR, а значит, наша кривая в идеале должна стремиться к точке (0,1).

**Задания:** 
1. Получить вероятности принадлежности наблюдений к классу 1 по обученной модели линейной регресии. 
2. Реализовать функции для оценки качетсва полученной модели по различным метрикам. 
3. Построить кривую ROC (отобразить график кривой).

In [10]:
# 1. Получить вероятности принадлежности наблюдений к классу 1.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Выделяем признаки и переводим метки из {-1, 1} в {0, 1}.
X = df[iris.feature_names].to_numpy()
y = (df['target'] == 1).astype(int).to_numpy()

# Делим данные на обучающую и тестовую части.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Обучаем модель и получаем вероятность класса 1 на тестовых объектах.
model = LogisticRegression()
model.fit(X_train, y_train)
y_probability = model.predict_proba(X_test)[:, 1]

probability_table = pd.DataFrame({
    'Истинный класс': y_test,
    'Вероятность класса 1': y_probability
})
probability_table.head(10)

,Истинный класс,Вероятность класса 1
0,0,0.007821
1,0,0.001280
2,0,0.002492
3,0,0.005159
4,0,0.000496
5,0,0.000604
6,1,0.967809
7,1,0.959836
8,1,0.974370
9,1,0.986470


In [14]:
# 2. Реализовать функции для оценки качества модели по различным метрикам.
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score
)

def calculate_metrics(y_true, y_probability, threshold=0.5):
    y_predicted = (y_probability >= threshold).astype(int)

    true_positive = np.sum((y_true == 1) & (y_predicted == 1))
    true_negative = np.sum((y_true == 0) & (y_predicted == 0))
    false_positive = np.sum((y_true == 0) & (y_predicted == 1))
    false_negative = np.sum((y_true == 1) & (y_predicted == 0))

    accuracy = (true_positive + true_negative) / (true_positive + true_negative + false_positive + false_negative)
    precision = true_positive / (true_positive + false_positive)
    recall = true_positive / (true_positive + false_negative)
    f1 = 2 * precision * recall / (precision + recall)

    log_loss_value = -np.mean(
        y_true * np.log(y_probability)
        + (1 - y_true) * np.log(1 - y_probability))

    positive_count = np.sum(y_true == 1)
    negative_count = np.sum(y_true == 0)
    thresholds = np.r_[np.inf, np.sort(np.unique(y_probability))[::-1], -np.inf]
    false_positive_rates = []
    true_positive_rates = []
    for current_threshold in thresholds:
        predicted = y_probability >= current_threshold
        false_positive_rates.append(np.sum((y_true == 0) & predicted) / negative_count)
        true_positive_rates.append(np.sum((y_true == 1) & predicted) / positive_count)
    roc_auc = sum(
        (false_positive_rates[index] - false_positive_rates[index - 1])
        * (true_positive_rates[index] + true_positive_rates[index - 1]) / 2
        for index in range(1, len(false_positive_rates)))

    return {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'Log loss': log_loss_value,
        'ROC-AUC': roc_auc
    }

metrics = calculate_metrics(y_test, y_probability)
pd.Series(metrics, name='Значение метрики').round(3)

Accuracy     1.000
Precision    1.000
Recall       1.000
F1           1.000
Log loss     0.014
ROC-AUC      1.000
Name: Значение метрики, dtype: float64

In [15]:
# 3. Построить кривую ROC (отобразить график кривой).
from sklearn.metrics import roc_curve

false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_probability)
roc_auc = roc_auc_score(y_test, y_probability)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=false_positive_rate,
    y=true_positive_rate,
    name=f'ROC (AUC = {roc_auc:.3f})',
    showlegend=True
))

fig_roc.update_layout(
    title='ROC-кривая',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    xaxis={'range': [0, 1]},
    yaxis={'range': [0, 1]}
)
fig_roc.show()